# ЛР №3. Оценка качества моделей машинного обучения. <br>ЛР №5. Градиентные методы в решении задач машинного обучения

## 0. Загрузка библиотек и предобработанных данных из ЛР №1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.datasets import load_digits, fetch_california_housing
import warnings
warnings.filterwarnings('ignore')

print("Библиотеки загружены успешно.")

## I. Бинарная классификация

### 1. Загрузка предобработанных данных Титаника из ЛР №1

In [ ]:
# Загружаем предобработанный датасет из 1-й лабораторной работы
# Если файл не найден — воспроизводим предобработку прямо здесь
try:
    df = pd.read_csv('train_processed.csv')
    print("Файл train_processed.csv загружен успешно.")
except FileNotFoundError:
    print("Файл не найден. Воспроизводим предобработку из ЛР №1...")
    from sklearn.preprocessing import LabelEncoder
    import pandas as pd

    df = pd.read_csv('https://raw.githubusercontent.com/josem279/titanic_dataset/refs/heads/main/data/train.csv')

    # Извлечение титула и заполнение пропусков Age
    df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
    title_age_medians = df.groupby('Title')['Age'].median()
    df['Age'] = df['Age'].fillna(df['Title'].map(title_age_medians))
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Cabin'] = df['Cabin'].fillna('U')
    most_frequent_port = df['Embarked'].mode()[0]
    df['Embarked'] = df['Embarked'].fillna(most_frequent_port)

    # Кодирование
    le = LabelEncoder()
    df['Sex_encoded'] = le.fit_transform(df['Sex'])
    df = pd.get_dummies(df, columns=['Embarked'], prefix='Embarked', drop_first=False)
    df['Deck'] = df['Cabin'].str[0]
    df['Deck_encoded'] = LabelEncoder().fit_transform(df['Deck'])

    title_mapping = {'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master'}
    df['Title_grouped'] = df['Title'].map(title_mapping).fillna('Other')
    df['Title_encoded'] = LabelEncoder().fit_transform(df['Title_grouped'])

    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    bins = [0, 12, 18, 35, 60, 100]
    labels = ['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']
    df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
    df['AgeGroup_encoded'] = LabelEncoder().fit_transform(df['AgeGroup'].astype(str))

    scaler_tmp = StandardScaler()
    features_to_scale = ['Age', 'Fare', 'FamilySize']
    df[[f + '_scaled' for f in features_to_scale]] = scaler_tmp.fit_transform(df[features_to_scale])

    df['Survived'] = pd.to_numeric(df['Survived'], errors='coerce')
    print("Предобработка завершена.")

print(f"Размер датасета: {df.shape}")
display(df.head(3))

### 2. Формирование матрицы признаков и разделение на train/test

In [ ]:
# Набор признаков для классификации
feature_cols = [
    'Pclass', 'Sex_encoded', 'Age_scaled', 'Fare_scaled',
    'FamilySize_scaled', 'IsAlone', 'Title_encoded', 'AgeGroup_encoded',
    'Deck_encoded'
]
# Добавляем Embarked-столбцы если они есть
embarked_cols = [c for c in df.columns if c.startswith('Embarked_')]
feature_cols += embarked_cols

X = df[feature_cols].copy()
y = df['Survived'].astype(int)

print(f"Признаки: {feature_cols}")
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Распределение целевой переменной:\n{y.value_counts()}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nТренировочная: {X_train.shape[0]}, Тестовая: {X_test.shape[0]}")

### 3. Логистическая регрессия (sklearn)

In [ ]:
# LogisticRegressionCV автоматически подбирает параметр регуляризации C
lr_cv = LogisticRegressionCV(
    Cs=10,            # 10 значений C для поиска
    cv=5,             # 5-кратная кросс-валидация
    max_iter=1000,
    random_state=42,
    scoring='accuracy'
)
lr_cv.fit(X_train, y_train)

print(f"Лучшее C (параметр регуляризации): {lr_cv.C_[0]:.4f}")

y_pred_lr = lr_cv.predict(X_test)

print("\n=== Logistic Regression (LogisticRegressionCV) ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_lr):.4f}")
print()
print(classification_report(y_test, y_pred_lr, target_names=['Не выжил', 'Выжил']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_lr)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Не выжил', 'Выжил'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()

In [ ]:
# Сравниваем несколько значений регуляризации
print("Влияние параметра регуляризации C на качество (кросс-валидация 5-fold):")
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
cv_means, cv_stds = [], []

for C in C_values:
    lr_tmp = LogisticRegression(C=C, max_iter=1000, random_state=42)
    scores = cross_val_score(lr_tmp, X_train, y_train, cv=5, scoring='accuracy')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())
    print(f"  C={C:<6} | CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

plt.figure(figsize=(8, 4))
plt.errorbar(range(len(C_values)), cv_means, yerr=cv_stds, marker='o', capsize=5)
plt.xticks(range(len(C_values)), [str(c) for c in C_values])
plt.xlabel("Параметр C (обратная сила регуляризации)")
plt.ylabel("CV Accuracy")
plt.title("Зависимость качества LogReg от параметра регуляризации C")
plt.grid(True)
plt.tight_layout()
plt.show()

### 4. (*) Собственная реализация логистической регрессии через градиентный спуск (ЛР №6)

Логистическая регрессия обучается минимизацией log-loss:

$$L(w) = -\frac{1}{m}\sum_{i=1}^{m}\left[y^{(i)}\log\sigma(x^{(i)} w) + (1-y^{(i)})\log(1-\sigma(x^{(i)} w))\right]$$

Обновление весов на каждом шаге:

$$w \leftarrow w - \alpha \cdot \nabla L(w)$$

In [ ]:
class LogisticRegressionGD:
    """
    Логистическая регрессия, обученная через градиентный спуск (batch).
    """
    def __init__(self, lr=0.1, n_iter=1000, tol=1e-6):
        self.lr = lr          # скорость обучения
        self.n_iter = n_iter  # максимум итераций
        self.tol = tol        # порог остановки
        self.losses_ = []

    @staticmethod
    def _sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        X = np.array(X, dtype=float)
        y = np.array(y, dtype=float)
        m, n = X.shape

        # Инициализация весов нулями
        self.w_ = np.zeros(n)
        self.b_ = 0.0

        for _ in range(self.n_iter):
            z = X @ self.w_ + self.b_
            y_hat = self._sigmoid(z)

            # Градиенты
            error = y_hat - y
            dw = (X.T @ error) / m
            db = error.mean()

            # Обновление
            self.w_ -= self.lr * dw
            self.b_ -= self.lr * db

            # Log-loss
            eps = 1e-15
            loss = -np.mean(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))
            self.losses_.append(loss)

            if len(self.losses_) > 1 and abs(self.losses_[-2] - loss) < self.tol:
                print(f"  Сошлось на итерации {_ + 1}")
                break
        return self

    def predict_proba(self, X):
        X = np.array(X, dtype=float)
        return self._sigmoid(X @ self.w_ + self.b_)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


# Обучение
gd_model = LogisticRegressionGD(lr=0.1, n_iter=2000)
X_train_arr = X_train.values
X_test_arr  = X_test.values
gd_model.fit(X_train_arr, y_train.values)

y_pred_gd = gd_model.predict(X_test_arr)

print("=== Логистическая регрессия (собственный градиентный спуск) ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_gd):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_gd):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_gd):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_gd):.4f}")
print()
print(classification_report(y_test, y_pred_gd, target_names=['Не выжил', 'Выжил']))

In [ ]:
# Кривая обучения (loss)
plt.figure(figsize=(8, 4))
plt.plot(gd_model.losses_, color='steelblue')
plt.xlabel("Итерация")
plt.ylabel("Log-Loss")
plt.title("Кривая обучения — собственный градиентный спуск")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Сравнение sklearn vs собственная реализация
print("=== Сравнение результатов ===")
print(f"{'Метрика':<12} {'sklearn LogReg':>16} {'Собственный GD':>16}")
print("-" * 46)
for metric_name, fn in [("Accuracy", accuracy_score), ("Precision", precision_score),
                          ("Recall", recall_score), ("F1", f1_score)]:
    sk_val  = fn(y_test, y_pred_lr) if metric_name == "Accuracy" else fn(y_test, y_pred_lr)
    gd_val  = fn(y_test, y_pred_gd) if metric_name == "Accuracy" else fn(y_test, y_pred_gd)
    print(f"{metric_name:<12} {sk_val:>16.4f} {gd_val:>16.4f}")

---
## II. Многоклассовая классификация (kNN на датасете Digits)

In [ ]:
# Загрузка датасета рукописных цифр
digits = load_digits()
X_dig, y_dig = digits.data, digits.target

print(f"Размер датасета: {X_dig.shape}")
print(f"Классы: {np.unique(y_dig)}")
print(f"Размер одного изображения: 8×8 пикселей")

In [ ]:
# Визуализация примеров из датасета
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for digit in range(10):
    idx = np.where(y_dig == digit)[0][0]
    axes[0, digit].imshow(digits.images[idx], cmap='gray_r')
    axes[0, digit].set_title(f"Цифра {digit}")
    axes[0, digit].axis('off')

    idx2 = np.where(y_dig == digit)[0][1]
    axes[1, digit].imshow(digits.images[idx2], cmap='gray_r')
    axes[1, digit].axis('off')

plt.suptitle("Примеры рукописных цифр (два образца каждой)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Разделение и масштабирование
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_dig, y_dig, test_size=0.2, random_state=42, stratify=y_dig
)

scaler_d = StandardScaler()
X_train_d_sc = scaler_d.fit_transform(X_train_d)
X_test_d_sc  = scaler_d.transform(X_test_d)

print(f"Train: {X_train_d.shape}, Test: {X_test_d.shape}")

### 5. Подбор оптимального k методом Grid Search

In [ ]:
param_grid = {'n_neighbors': list(range(1, 21))}

knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
knn_grid.fit(X_train_d_sc, y_train_d)

best_k = knn_grid.best_params_['n_neighbors']
best_cv_score = knn_grid.best_score_
print(f"Лучший k = {best_k}, CV Accuracy = {best_cv_score:.4f}")

In [ ]:
# Кривая: CV-точность vs k
cv_results = knn_grid.cv_results_
mean_scores = cv_results['mean_test_score']
std_scores  = cv_results['std_test_score']
k_values    = param_grid['n_neighbors']

plt.figure(figsize=(9, 4))
plt.plot(k_values, mean_scores, marker='o', label='CV Accuracy')
plt.fill_between(k_values,
                 mean_scores - std_scores,
                 mean_scores + std_scores, alpha=0.2)
plt.axvline(best_k, color='red', linestyle='--', label=f'Лучший k={best_k}')
plt.xlabel("k (число соседей)")
plt.ylabel("CV Accuracy")
plt.title("Зависимость точности kNN от числа соседей (Digits)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### 6. Качество классификации с оптимальным k (ЛР №4)

In [ ]:
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_d_sc, y_train_d)
y_pred_knn = knn_best.predict(X_test_d_sc)

print(f"=== kNN (k={best_k}) — Digits ===")
print(f"Accuracy:  {accuracy_score(y_test_d, y_pred_knn):.4f}")
print(f"Macro F1:  {f1_score(y_test_d, y_pred_knn, average='macro'):.4f}")
print()
print(classification_report(y_test_d, y_pred_knn))

In [ ]:
# Confusion Matrix для многоклассового случая
cm_d = confusion_matrix(y_test_d, y_pred_knn)
plt.figure(figsize=(9, 7))
sns.heatmap(cm_d, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel("Предсказанная цифра")
plt.ylabel("Истинная цифра")
plt.title(f"Confusion Matrix — kNN (k={best_k})")
plt.tight_layout()
plt.show()

In [ ]:
# Визуализация ошибочных предсказаний
wrong_idx = np.where(y_pred_knn != y_test_d)[0]
print(f"Всего ошибок: {len(wrong_idx)} из {len(y_test_d)}")

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    if i < len(wrong_idx):
        idx = wrong_idx[i]
        img = X_test_d[idx].reshape(8, 8)
        ax.imshow(img, cmap='gray_r')
        ax.set_title(f"И:{y_test_d[idx]} П:{y_pred_knn[idx]}", fontsize=8)
    ax.axis('off')
plt.suptitle("Ошибочные предсказания kNN (И=истина, П=предсказание)", y=1.02)
plt.tight_layout()
plt.show()

---
## III. Регрессия (California Housing)

In [ ]:
# Загрузка датасета
housing = fetch_california_housing()
X_h = pd.DataFrame(housing.data, columns=housing.feature_names)
y_h = housing.target  # средняя цена дома (в $100 000)

print("Описание датасета:")
print(housing.DESCR[:800])
print(f"\nРазмер: {X_h.shape}")
display(X_h.describe())

In [ ]:
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42
)

scaler_h = StandardScaler()
X_train_h_sc = scaler_h.fit_transform(X_train_h)
X_test_h_sc  = scaler_h.transform(X_test_h)

print(f"Train: {X_train_h.shape}, Test: {X_test_h.shape}")

### 7. Линейная регрессия и её разновидности

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

models = {
    'LinearRegression (без регуляризации)': LinearRegression(),
    'Ridge (L2, alpha=1.0)':               Ridge(alpha=1.0),
    'Lasso (L1, alpha=0.01)':              Lasso(alpha=0.01, max_iter=5000),
    'ElasticNet (alpha=0.01)':             ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000),
}

results_h = {}
for name, model in models.items():
    model.fit(X_train_h_sc, y_train_h)
    y_pred_h = model.predict(X_test_h_sc)
    mse  = mean_squared_error(y_test_h, y_pred_h)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_test_h, y_pred_h)
    r2   = r2_score(y_test_h, y_pred_h)
    results_h[name] = {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R²': r2}
    print(f"\n=== {name} ===")
    print(f"  MSE:  {mse:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  R²:   {r2:.4f}")

In [ ]:
# Визуальное сравнение моделей регрессии
results_df = pd.DataFrame(results_h).T
display(results_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

results_df[['RMSE', 'MAE']].plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title("RMSE и MAE — сравнение моделей")
axes[0].set_ylabel("Ошибка")
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(axis='y', alpha=0.5)

results_df[['R²']].plot(kind='bar', ax=axes[1], color='mediumseagreen')
axes[1].set_title("R² — сравнение моделей")
axes[1].set_ylabel("R²")
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='y', alpha=0.5)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Предсказание vs реальные значения (лучшая модель)
best_model_name = max(results_h, key=lambda k: results_h[k]['R²'])
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test_h_sc)

plt.figure(figsize=(7, 6))
plt.scatter(y_test_h, y_pred_best, alpha=0.3, s=10, color='steelblue')
plt.plot([y_test_h.min(), y_test_h.max()],
         [y_test_h.min(), y_test_h.max()], 'r--', lw=2)
plt.xlabel("Реальные значения")
plt.ylabel("Предсказанные значения")
plt.title(f"Предсказание vs Реальность\n({best_model_name})")
plt.tight_layout()
plt.show()

In [ ]:
# Влияние параметра alpha на Ridge-регрессию
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
ridge_r2, ridge_rmse = [], []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_h_sc, y_train_h)
    preds = ridge.predict(X_test_h_sc)
    ridge_r2.append(r2_score(y_test_h, preds))
    ridge_rmse.append(np.sqrt(mean_squared_error(y_test_h, preds)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogx(alphas, ridge_r2, marker='o', color='steelblue')
axes[0].set_xlabel("alpha"); axes[0].set_ylabel("R²"); axes[0].set_title("Ridge: R² vs alpha"); axes[0].grid(True)
axes[1].semilogx(alphas, ridge_rmse, marker='o', color='coral')
axes[1].set_xlabel("alpha"); axes[1].set_ylabel("RMSE"); axes[1].set_title("Ridge: RMSE vs alpha"); axes[1].grid(True)
plt.tight_layout()
plt.show()

---
## Контрольные вопросы

### ЛР №4 — Оценка качества

**1. Какие метрики используются для оценки качества классификатора?**

Accuracy (доля правильных ответов), Precision (точность = TP/(TP+FP)), Recall (полнота = TP/(TP+FN)), F1-score (гармоническое среднее P и R), ROC-AUC. Для многоклассового случая используются macro/micro/weighted варианты.

**2. Что такое confusion matrix и как её интерпретировать?**

Матрица ошибок — таблица, строки которой соответствуют истинным классам, а столбцы — предсказанным. Диагональ — правильные предсказания, остальные ячейки — ошибки (FP, FN).

**3. Что такое кросс-валидация? Зачем она нужна?**

Метод оценки модели: данные делятся на k блоков; k-1 используются для обучения, 1 — для проверки; процедура повторяется k раз. Снижает дисперсию оценки качества, использует все данные и для обучения, и для валидации.

**4. Для чего нужна регуляризация?**

Регуляризация (L1/Lasso, L2/Ridge) добавляет штраф за большие веса, препятствуя переобучению. L1 даёт разреженные веса (feature selection), L2 уменьшает все веса равномерно.

**5. Как работает метрика R² для регрессии?**

R² = 1 − (SS_res / SS_tot). Показывает долю дисперсии целевой переменной, объяснённую моделью. R²=1 — идеально; R²=0 — модель не лучше среднего.

---

### ЛР №5 — Многоклассовая классификация (kNN)

**1. Как работает метод k ближайших соседей?**

Для нового объекта находим k ближайших (по метрике, напр. Евклидовой) обучающих объектов и выбираем большинством голосов класс. Параметр k — гиперпараметр.

**2. Как выбрать оптимальное k?**

С помощью кросс-валидации или GridSearchCV: перебираем значения k на валидации и выбираем то, при котором выбранная метрика максимальна.

**3. Почему важно масштабировать признаки для kNN?**

kNN основан на расстояниях. Признаки с большим масштабом (напр., Fare vs Age) будут доминировать. StandardScaler приводит признаки к единому масштабу.

---

### ЛР №6 — Градиентный спуск

**1. Что такое градиентный спуск?**

Итерационный алгоритм минимизации функции потерь: w ← w − α·∇L(w). На каждом шаге двигаемся в сторону антиградиента.

**2. Какова роль скорости обучения (learning rate)?**

При слишком большой α — шаги большие, алгоритм может расходиться. При слишком маленькой α — сходимость медленная. Выбирается подбором (learning rate schedule, Adam и т.д.).

**3. Чем стохастический (SGD) отличается от батчевого (Batch GD)?**

Batch GD вычисляет градиент по всем объектам (точен, но медленен). SGD — по одному случайному объекту (быстро, но шумно). Mini-batch GD — компромисс (по k объектам).

**4. Что такое сигмоид и зачем он нужен в логистической регрессии?**

σ(z) = 1/(1+e^{-z}) — переводит любое вещественное число в вероятность ∈ (0,1). Логистическая регрессия предсказывает P(y=1|x) = σ(w·x + b).